In [ ]:
# ¿Qué evidencia de preferencias contiene FilmTrust y qué tan dispersa está?

import numpy as np
import pandas as pd

raw_ratings = pd.read_csv("../data/filmtrust_ratings.csv")
ratings = raw_ratings.groupby(["user", "movie"], as_index=False)["score"].mean()
user_item = ratings.pivot(index="user", columns="movie", values="score")
density = len(ratings) / (user_item.shape[0] * user_item.shape[1])
pd.DataFrame({"raw_ratings": [len(raw_ratings)], "consolidated_ratings": [len(ratings)], "users": [user_item.shape[0]], "movies": [user_item.shape[1]], "matrix_density": [density]})

In [ ]:
# ¿Qué usuarios aportan suficiente historial para encontrar preferencias comparables?

user_activity = ratings.groupby("user").agg(
    ratings=("score", "size"),
    mean_score=("score", "mean"),
).sort_values("ratings", ascending=False)
user_activity.head(10)

In [ ]:
# ¿Cómo se comparan preferencias sin confundir una ausencia de calificación con una baja calificación?

def cosine_similarity(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    normalized = np.divide(matrix, norms, out=np.zeros_like(matrix, dtype=float), where=norms != 0)
    return normalized @ normalized.T

target_user = 272
minimum_common_ratings = 2
minimum_similarity = 0.05
observed = user_item.notna().astype(int)
user_means = user_item.mean(axis=1)
centered_ratings = user_item.sub(user_means, axis=0).fillna(0)

In [ ]:
# ¿Qué vecinos de la persona 272 tienen al menos dos películas en común y afinidad positiva?

similarity = pd.DataFrame(
    cosine_similarity(centered_ratings.to_numpy()),
    index=user_item.index,
    columns=user_item.index,
)
common_ratings = pd.DataFrame(
    observed.to_numpy() @ observed.to_numpy().T,
    index=user_item.index,
    columns=user_item.index,
)
neighbors = pd.DataFrame({
    "neighbor_user": similarity.index,
    "similarity": similarity.loc[target_user].to_numpy(),
    "common_ratings": common_ratings.loc[target_user].to_numpy(),
    "neighbor_ratings": observed.sum(axis=1).to_numpy(),
}).query("neighbor_user != @target_user")
eligible_neighbors = neighbors.query(
    "common_ratings >= @minimum_common_ratings and similarity >= @minimum_similarity"
).sort_values("similarity", ascending=False)
eligible_neighbors.head(10)

In [ ]:
# ¿Qué películas no valoradas tienen una estimación respaldada por al menos dos vecinos?

weighted_deviations = pd.Series(0.0, index=user_item.columns)
weight_sums = pd.Series(0.0, index=user_item.columns)
support_counts = pd.Series(0, index=user_item.columns, dtype=int)
for neighbor in eligible_neighbors.itertuples(index=False):
    deviations = user_item.loc[neighbor.neighbor_user] - user_means.loc[neighbor.neighbor_user]
    available = deviations.notna()
    weighted_deviations.loc[available] += neighbor.similarity * deviations.loc[available]
    weight_sums.loc[available] += neighbor.similarity
    support_counts.loc[available] += 1

predictions = user_means.loc[target_user] + weighted_deviations.divide(weight_sums.where(weight_sums != 0))
recommendations = pd.DataFrame({
    "movie_id": predictions.index,
    "predicted_score": predictions.values,
    "supporting_neighbors": support_counts.values,
})
recommendations = recommendations.loc[
    user_item.loc[target_user].isna().to_numpy()
    & recommendations["predicted_score"].notna()
    & recommendations["supporting_neighbors"].ge(2)
] .sort_values(["predicted_score", "supporting_neighbors"], ascending=False)
recommendations.head(10)

In [ ]:
# Se conservan la cobertura, los vecinos seleccionados y las recomendaciones como evidencia del taller.

from pathlib import Path

submission_dir = Path("../submission")
coverage = pd.DataFrame([
    {"metric": "raw_ratings", "value": len(raw_ratings)},
    {"metric": "consolidated_ratings", "value": len(ratings)},
    {"metric": "users", "value": user_item.shape[0]},
    {"metric": "movies", "value": user_item.shape[1]},
    {"metric": "matrix_density", "value": density},
    {"metric": "target_user", "value": target_user},
    {"metric": "target_ratings", "value": user_item.loc[target_user].notna().sum()},
    {"metric": "eligible_neighbors", "value": len(eligible_neighbors)},
])
coverage.to_csv(submission_dir / "coverage_summary.csv", index=False)
eligible_neighbors.to_csv(submission_dir / "nearest_neighbors.csv", index=False)
recommendations.to_csv(submission_dir / "recommendations.csv", index=False)